# Notebook 01 — Data Collection for CAMP

**Project:** Calibration-Aware ML-Predictive Scheduling for Single-GPU LLM Inference  
**Environment:** Google Colab + NVIDIA Tesla T4  
**LLM:** `Qwen/Qwen2.5-1.5B-Instruct` served by vLLM

## What this notebook does

1. Checks the Colab GPU and installs the required runtime packages (only if missing).
2. Starts Qwen through vLLM's OpenAI-compatible local server.
3. Generates steady, mixed and bursty request workloads.
4. Collects real request-level timings and GPU telemetry.
5. Saves two files:
   - `latency_dataset.csv` — measured examples for training the latency predictor.
   - `benchmark_traces.csv` — separate, not-yet-executed requests for the later four-scheduler comparison.

**No GUI, dashboard or separate application is used.** Results appear directly below the notebook cells.

> This is the initial pilot collection. After validating the pipeline, increase the run sizes/repetitions before treating the data as final research evidence. Do not report illustrative or pilot metrics as final results.

## Cell 1 — Check the GPU

Select **Runtime → Change runtime type → T4 GPU** before running this notebook.

In [ ]:
import subprocess, platform, torch, sys

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("GPU is not enabled. In Colab, choose Runtime > Change runtime type > T4 GPU.")

print("GPU:", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))
print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print(subprocess.check_output(["nvidia-smi"], text=True))

Python: 3.13.16
PyTorch: 2.14.0+cu130
CUDA available: True
GPU: Tesla T4
Compute capability: (7, 5)
GPU memory (GiB): 14.56
Mon Nov 23 08:01:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|               

## Cell 2 — Install packages if needed

The vLLM CUDA 13.0 installation command below follows the current vLLM GPU installation documentation. If vLLM is already installed, this cell skips reinstalling it. If it installs vLLM now, **restart the Colab runtime once**, then continue from Cell 1.

In [ ]:
import importlib.util, subprocess, sys

if importlib.util.find_spec("vllm") is None:
    print("Installing vLLM for CUDA 13.0. This may take several minutes...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "vllm",
        "--extra-index-url", "https://download.pytorch.org/whl/cu130"
    ])
    print("vLLM installation finished. Restart the Colab runtime, then rerun this notebook.")
else:
    print("vLLM is already installed; skipping installation.")

missing = [p for p in ["openai", "transformers", "pandas", "requests"]
           if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("Installed:", ", ".join(missing))
else:
    print("Client/data packages are already installed.")

vLLM is already installed; skipping installation.
Client/data packages are already installed.


## Cell 3 — Configure and start the Qwen vLLM server

The pilot uses FP16, a 2,048-token context limit and at most four simultaneous sequences to stay conservative on the T4. If startup fails, inspect the printed log before changing settings.

In [ ]:
import os, sys, time, shutil, subprocess, requests
from pathlib import Path

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8000
BASE_URL = f"http://{SERVER_HOST}:{SERVER_PORT}/v1"
SERVER_LOG_PATH = "/content/vllm_server.log"

# Reuse an already-running local server if it responds to the health check.
server_ready = False
try:
    server_ready = requests.get(f"http://{SERVER_HOST}:{SERVER_PORT}/health", timeout=3).status_code == 200
except Exception:
    server_ready = False

server_process = None
server_log_handle = None

if not server_ready:
    vllm_executable = shutil.which("vllm")
    if not vllm_executable:
        raise RuntimeError("The vLLM command was not found. Run the installation cell, restart the runtime, and try again.")

    command = [
        vllm_executable, "serve", MODEL_ID,
        "--dtype", "half",
        "--max-model-len", "2048",
        "--max-num-seqs", "4",
        "--gpu-memory-utilization", "0.85",
        "--host", SERVER_HOST,
        "--port", str(SERVER_PORT)
    ]
    server_log_handle = open(SERVER_LOG_PATH, "w")
    server_process = subprocess.Popen(command, stdout=server_log_handle, stderr=subprocess.STDOUT)
    print("Started vLLM. Waiting for model download and server startup...")

    deadline = time.time() + 900
    while time.time() < deadline:
        if server_process.poll() is not None:
            break
        try:
            r = requests.get(f"http://{SERVER_HOST}:{SERVER_PORT}/health", timeout=3)
            if r.status_code == 200:
                server_ready = True
                break
        except Exception:
            pass
        time.sleep(5)

if not server_ready:
    print("\n--- Last 100 lines of vLLM log ---")
    if Path(SERVER_LOG_PATH).exists():
        print("\n".join(Path(SERVER_LOG_PATH).read_text(errors="ignore").splitlines()[-100:]))
    raise RuntimeError("vLLM did not become ready. Check the log above before continuing.")

print("vLLM server is ready:", BASE_URL)
print("Server log:", SERVER_LOG_PATH)

Started vLLM. Waiting for model download and server startup...
vLLM server is ready: http://127.0.0.1:8000/v1
Server log: /content/vllm_server.log


## Cell 4 — Connect to Qwen and load its tokenizer

The tokenizer is used to record actual prompt/output token counts. vLLM exposes an OpenAI-compatible chat-completions endpoint.

In [ ]:
from openai import AsyncOpenAI
from transformers import AutoTokenizer

client = AsyncOpenAI(
    base_url=BASE_URL,
    api_key="EMPTY",
    timeout=600.0,
    max_retries=0
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True
)

# Check vLLM model availability
models = await client.models.list()

print("Available model IDs:")
for model in models.data:
    print("-", model.id)

# Test Qwen inference
smoke = await client.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {
            "role": "user",
            "content": "Explain LLM inference scheduling in simple terms."
        }
    ],
    max_tokens=40,
    temperature=0
)

print("\nQwen response:")
print(smoke.choices[0].message.content)

print("\nSUCCESS: Qwen inference is working!")

Available model IDs:
- Qwen/Qwen2.5-1.5B-Instruct

Qwen response:
LLM inference scheduling is the process of deciding which tasks or requests should be processed next when multiple models need to run simultaneously. It involves managing and coordinating the execution of different language model instances (LLMs

SUCCESS: Qwen inference is working!


## Cell 5 — Define pilot workloads and request prompts

We use three open-loop arrival patterns:

- **steady:** regular request arrivals.
- **mixed:** variable Poisson-style arrival gaps.
- **bursty:** short groups of requests followed by pauses.

Prompt lengths and requested output limits vary in every workload. The three repeats use client concurrency limits 1, 2 and 4, so the predictor sees different load levels. Each prompt includes a unique request ID to avoid repeatedly submitting identical prompt prefixes.

In [ ]:
import random, numpy as np, pandas as pd, json, uuid
from collections import deque

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Pilot settings. Increase these only after the first end-to-end run succeeds.
REQUESTS_PER_RUN = 20
REPEATS_PER_WORKLOAD = 3
CONCURRENCY_LEVELS = [1, 2, 4]  # T4 pilot; vLLM is configured with max-num-seqs=4
PROMPT_TARGET_CHOICES = [32, 128, 256, 512]
OUTPUT_LIMIT_CHOICES = [32, 64, 96]
WORKLOAD_TYPES = ["steady", "mixed", "bursty"]

TEXT_BANK = [
    "Large language model serving systems receive requests with different prompt sizes and generation limits.",
    "A request scheduler manages waiting jobs and decides when each request is sent to the inference engine.",
    "GPU memory is shared by model weights, intermediate activations and the key value cache used during decoding.",
    "The prefill phase processes input tokens while the decode phase generates output tokens sequentially.",
    "Throughput, tail latency and goodput are important measurements for an online inference service.",
    "A burst of requests can increase queue depth and change the latency experienced by later requests.",
    "An admission controller can limit the number of active requests without modifying the inference backend.",
    "A workload trace records arrival times, prompt lengths and requested output token limits for reproducible tests."
]

def build_prompt(target_tokens: int, request_id: str, rng: random.Random):
    """Build a varied prompt and return its actual chat-template token count."""
    pieces = [f"Request reference {request_id}."]
    content = " ".join(pieces)
    target_content_tokens = max(8, target_tokens - 20)
    while len(tokenizer.encode(content, add_special_tokens=False)) < target_content_tokens:
        pieces.append(rng.choice(TEXT_BANK))
        content = " ".join(pieces)
    content_ids = tokenizer.encode(content, add_special_tokens=False)[:target_content_tokens]
    content = tokenizer.decode(content_ids, skip_special_tokens=True)
    messages = [{"role": "user", "content": content + "\nGive a concise, relevant answer."}]
    prompt_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True)
    return messages, len(prompt_ids), content

def make_arrival_offsets(n: int, workload_type: str, rng: np.random.Generator):
    if workload_type == "steady":
        offsets = np.arange(n, dtype=float) * 0.60
    elif workload_type == "mixed":
        gaps = rng.exponential(scale=0.60, size=n)
        offsets = np.cumsum(gaps) - gaps[0]
    elif workload_type == "bursty":
        offsets = []
        for i in range(n):
            burst_number, within_burst = divmod(i, 5)
            offsets.append(burst_number * 3.0 + within_burst * 0.06)
        offsets = np.asarray(offsets, dtype=float)
    else:
        raise ValueError(f"Unknown workload type: {workload_type}")
    return offsets

def make_request_specs(workload_type: str, run_id: str, n: int, seed: int):
    py_rng = random.Random(seed)
    np_rng = np.random.default_rng(seed)
    offsets = make_arrival_offsets(n, workload_type, np_rng)
    specs = []
    for i in range(n):
        request_id = f"{run_id}_req_{i:04d}"
        target_prompt = py_rng.choice(PROMPT_TARGET_CHOICES)
        max_output = py_rng.choice(OUTPUT_LIMIT_CHOICES)
        messages, actual_prompt_tokens, prompt_text = build_prompt(target_prompt, request_id, py_rng)
        specs.append({
            "request_id": request_id,
            "run_id": run_id,
            "workload_type": workload_type,
            "arrival_offset_s": float(offsets[i]),
            "prompt_text": messages[0]["content"],
            "prompt_tokens": int(actual_prompt_tokens),
            "max_output_tokens": int(max_output)
        })
    return specs

print("Pilot configuration")
print("Requests per run:", REQUESTS_PER_RUN)
print("Repeats per workload:", REPEATS_PER_WORKLOAD)
print("Total measured requests:", REQUESTS_PER_RUN * REPEATS_PER_WORKLOAD * len(WORKLOAD_TYPES))
print("Concurrency levels across repeated runs:", CONCURRENCY_LEVELS)

Pilot configuration
Requests per run: 20
Repeats per workload: 3
Total measured requests: 180
Concurrency levels across repeated runs: [1, 2, 4]


## Cell 6 — Start GPU telemetry sampling

GPU utilisation and memory are sampled in the background approximately once per second. Each request uses the latest sample available at its admission time; this is an approximate telemetry snapshot, not a per-kernel measurement.

In [ ]:
import threading, time, subprocess

_gpu_stop = threading.Event()
_gpu_lock = threading.Lock()
_gpu_samples = []

def _gpu_sampler():
    while not _gpu_stop.is_set():
        try:
            result = subprocess.check_output([
                "nvidia-smi",
                "--query-gpu=utilization.gpu,memory.used,memory.total",
                "--format=csv,noheader,nounits"
            ], text=True, stderr=subprocess.DEVNULL, timeout=5)
            parts = [x.strip() for x in result.strip().split(",")]
            if len(parts) >= 3:
                sample = {
                    "sampled_at_perf": time.perf_counter(),
                    "gpu_utilization_pct": float(parts[0]),
                    "gpu_memory_used_mib": float(parts[1]),
                    "gpu_memory_total_mib": float(parts[2])
                }
                with _gpu_lock:
                    _gpu_samples.append(sample)
                    if len(_gpu_samples) > 5000:
                        del _gpu_samples[:1000]
        except Exception:
            pass
        _gpu_stop.wait(1.0)

def latest_gpu_sample():
    with _gpu_lock:
        return dict(_gpu_samples[-1]) if _gpu_samples else {
            "gpu_utilization_pct": np.nan,
            "gpu_memory_used_mib": np.nan,
            "gpu_memory_total_mib": np.nan
        }

_gpu_thread = threading.Thread(target=_gpu_sampler, daemon=True)
_gpu_thread.start()
time.sleep(2)
print("GPU telemetry sampler running. Latest sample:", latest_gpu_sample())

GPU telemetry sampler running. Latest sample: {'sampled_at_perf': 1480.903119611, 'gpu_utilization_pct': 0.0, 'gpu_memory_used_mib': 12381.0, 'gpu_memory_total_mib': 15360.0}


## Cell 7 — Collect real request timings

The client-side queue is open-loop: arrivals are scheduled from the trace and do not wait for earlier requests to finish. One, two or four workers (depending on the repeated run) dispatch requests to vLLM. For each request:

- `enqueue_time` is when it enters the external queue.
- `admission_time` is when a worker sends it to vLLM.
- `completion_time` is when the final streamed response chunk is received.
- `engine_latency_s = completion_time - admission_time`.
- `queue_wait_s = admission_time - enqueue_time`.

This follows the measurement boundary used in the supplied literature review. Since timestamps are observed by the local client/gateway, `engine_latency_s` includes local HTTP/client overhead and any internal vLLM queueing; it is not a GPU-kernel-only measurement.

In [ ]:
import asyncio, math
from collections import deque

async def collect_one_run(workload_type: str, run_id: str, specs: list, concurrency_limit: int):
    request_queue = asyncio.Queue()
    arrival_history = deque()
    rows = []
    active_requests = 0
    run_start = time.perf_counter()

    async def producer():
        for spec in specs:
            target = run_start + spec["arrival_offset_s"]
            delay = target - time.perf_counter()
            if delay > 0:
                await asyncio.sleep(delay)
            enqueue_at = time.perf_counter()
            arrival_history.append(enqueue_at)
            await request_queue.put((spec, enqueue_at))

    async def worker(worker_id: int):
        nonlocal active_requests
        while True:
            item = await request_queue.get()
            if item is None:
                request_queue.task_done()
                return

            spec, enqueue_at = item
            admission_at = time.perf_counter()
            queue_depth = request_queue.qsize()
            active_at_admission = active_requests
            while arrival_history and arrival_history[0] < admission_at - 5.0:
                arrival_history.popleft()
            window_elapsed = min(5.0, max(0.2, admission_at - run_start))
            arrival_rate = len(arrival_history) / window_elapsed
            gpu = latest_gpu_sample()
            active_requests += 1

            output_text = ""
            first_token_at = None
            status = "completed"
            error_message = ""
            completion_at = admission_at

            try:
                stream = await client.chat.completions.create(
                    model=MODEL_ID,
                    messages=[{"role": "user", "content": spec["prompt_text"]}],
                    max_tokens=spec["max_output_tokens"],
                    temperature=0,
                    stream=True
                )
                async for chunk in stream:
                    if not chunk.choices:
                        continue
                    piece = chunk.choices[0].delta.content
                    if piece:
                        if first_token_at is None:
                            first_token_at = time.perf_counter()
                        output_text += piece
                completion_at = time.perf_counter()
            except Exception as exc:
                completion_at = time.perf_counter()
                status = "failed"
                error_message = f"{type(exc).__name__}: {str(exc)[:300]}"
            finally:
                active_requests -= 1

            output_tokens = len(tokenizer.encode(output_text, add_special_tokens=False)) if output_text else 0
            rows.append({
                "request_id": spec["request_id"],
                "run_id": run_id,
                "workload_type": workload_type,
                "arrival_offset_s": spec["arrival_offset_s"],
                "prompt_text": spec["prompt_text"],
                "prompt_tokens": spec["prompt_tokens"],
                "max_output_tokens": spec["max_output_tokens"],
                "arrival_rate_5s_rps": arrival_rate,
                "queue_depth_at_admission": queue_depth,
                "active_requests_at_admission": active_at_admission,
                "concurrency_limit": concurrency_limit,
                "gpu_utilization_pct": gpu["gpu_utilization_pct"],
                "gpu_memory_used_mib": gpu["gpu_memory_used_mib"],
                "queue_wait_s": max(0.0, admission_at - enqueue_at),
                "engine_latency_s": max(0.0, completion_at - admission_at),
                "end_to_end_s": max(0.0, completion_at - enqueue_at),
                "ttft_engine_s": (first_token_at - admission_at) if first_token_at else np.nan,
                "actual_output_tokens": output_tokens,
                "status": status,
                "error": error_message,
                "output_preview": output_text[:220]
            })
            request_queue.task_done()

    workers = [asyncio.create_task(worker(i)) for i in range(concurrency_limit)]
    await producer()
    await request_queue.join()
    for _ in workers:
        await request_queue.put(None)
    await asyncio.gather(*workers)
    rows.sort(key=lambda x: x["request_id"])
    return rows

all_rows = []
for workload_type in WORKLOAD_TYPES:
    for repeat in range(REPEATS_PER_WORKLOAD):
        run_id = f"{workload_type}_run_{repeat+1}"
        seed = SEED + repeat * 101 + WORKLOAD_TYPES.index(workload_type) * 1009
        run_concurrency = CONCURRENCY_LEVELS[repeat % len(CONCURRENCY_LEVELS)]
        specs = make_request_specs(workload_type, run_id, REQUESTS_PER_RUN, seed)
        print(f"\nCollecting {run_id}: {len(specs)} requests at concurrency={run_concurrency}")
        run_rows = await collect_one_run(workload_type, run_id, specs, run_concurrency)
        all_rows.extend(run_rows)
        completed = sum(r["status"] == "completed" for r in run_rows)
        print(f"Completed: {completed}/{len(run_rows)}")

latency_df = pd.DataFrame(all_rows)
print("\nCollected rows:", len(latency_df))
print("Successful:", int((latency_df.status == "completed").sum()))
print("Failed:", int((latency_df.status != "completed").sum()))
display(latency_df.head(10))


Completed: 20/20

Completed: 20/20

Completed: 20/20

Completed: 20/20

Completed: 20/20

Completed: 20/20

Completed: 20/20

Completed: 20/20

Completed: 20/20

Collected rows: 180
Successful: 180
Failed: 0


,request_id,run_id,workload_type,arrival_offset_s,prompt_text,prompt_tokens,max_output_tokens,arrival_rate_5s_rps,queue_depth_at_admission,active_requests_at_admission,...,gpu_utilization_pct,gpu_memory_used_mib,queue_wait_s,engine_latency_s,end_to_end_s,ttft_engine_s,actual_output_tokens,status,error,output_preview
0,steady_run_1_req_0000,steady_run_1,steady,0.0,Request reference steady_run_1_req_0000\nGive ...,2,32,5.000000,0,0,...,0.0,12381.0,0.000399,0.585622,0.586022,0.090379,32,completed,,"I'm sorry, but I don't have enough context to ..."
1,steady_run_1_req_0001,steady_run_1,steady,0.6,Request reference steady_run_1_req_0001. The p...,2,32,3.324680,0,0,...,100.0,12381.0,0.000053,0.541665,0.541718,0.077919,32,completed,,The system described involves managing large l...
2,steady_run_1_req_0002,steady_run_1,steady,1.2,Request reference steady_run_1_req_0002. Throu...,2,96,2.498401,0,0,...,100.0,12381.0,0.000033,1.604761,1.604794,0.130931,96,completed,,"Throughput, tail latency, and goodput are cruc..."
3,steady_run_1_req_0003,steady_run_1,steady,1.8,Request reference steady_run_1_req_0003. An ad...,2,32,1.781632,1,0,...,100.0,12425.0,1.005651,0.510082,1.515733,0.047029,32,completed,,An admission controller in an online inference...
4,steady_run_1_req_0004,steady_run_1,steady,2.4,Request reference steady_run_1_req_0004. Throu...,2,32,1.808815,1,0,...,100.0,12425.0,0.915412,0.543611,1.459023,0.079247,32,completed,,"The throughput, tail latency, and goodput metr..."
5,steady_run_1_req_0005,steady_run_1,steady,3.0,Request reference steady_run_1_req_0005. A req...,2,64,1.812890,1,0,...,100.0,12425.0,0.859937,1.113457,1.973394,0.135394,64,completed,,The system described involves managing request...
6,steady_run_1_req_0006,steady_run_1,steady,3.6,Request reference steady_run_1_req_0006. Large...,2,96,1.808882,2,0,...,100.0,12425.0,1.374628,1.220993,2.595621,0.118817,73,completed,,A request scheduler in large language model se...
7,steady_run_1_req_0007,steady_run_1,steady,4.2,Request reference steady_run_1_req_0007. The p...,2,96,1.800000,3,0,...,100.0,12425.0,1.996497,1.354042,3.350538,0.129417,81,completed,,The prefill phase in large language model serv...
8,steady_run_1_req_0008,steady_run_1,steady,4.8,Request reference steady_run_1_req_0008\nGive ...,2,32,1.600000,4,0,...,100.0,12425.0,2.751943,0.528774,3.280717,0.070834,32,completed,,"I'm sorry, but I cannot provide the requested ..."
9,steady_run_1_req_0009,steady_run_1,steady,5.4,Request reference steady_run_1_req_0009. GPU m...,2,32,1.600000,4,0,...,100.0,12425.0,2.678434,0.625654,3.304088,0.143096,32,completed,,A request scheduler manages waiting jobs and d...


## Cell 8 — Create a separate benchmark trace

These requests are generated with independent seeds and have **not** been executed. Notebook 03 can replay the exact same trace against FIFO, Fixed-K, Heuristic Adaptive and CAMP ML-Predictive, keeping the workload identical across policies.

In [ ]:
BENCHMARK_REQUESTS_PER_WORKLOAD = 12
benchmark_rows = []
for workload_type in WORKLOAD_TYPES:
    specs = make_request_specs(
        workload_type,
        run_id=f"benchmark_{workload_type}",
        n=BENCHMARK_REQUESTS_PER_WORKLOAD,
        seed=SEED + 50000 + WORKLOAD_TYPES.index(workload_type) * 777
    )
    for spec in specs:
        benchmark_rows.append({
            "request_id": spec["request_id"],
            "workload_type": spec["workload_type"],
            "arrival_offset_s": spec["arrival_offset_s"],
            "prompt_text": spec["prompt_text"],
            "prompt_tokens": spec["prompt_tokens"],
            "max_output_tokens": spec["max_output_tokens"]
        })

benchmark_df = pd.DataFrame(benchmark_rows)
print("Independent benchmark trace rows:", len(benchmark_df))
display(benchmark_df.groupby("workload_type").agg(
    requests=("request_id", "count"),
    mean_prompt_tokens=("prompt_tokens", "mean"),
    mean_max_output=("max_output_tokens", "mean")
).round(2))

Independent benchmark trace rows: 36


,requests,mean_prompt_tokens,mean_max_output
workload_type,,,
bursty,12,2.0,74.67
mixed,12,2.0,72.00
steady,12,2.0,66.67


## Cell 9 — Save the two CSV files and inspect collection quality

`latency_dataset.csv` is the labelled latency dataset. `benchmark_traces.csv` is held out for policy benchmarking.

In [ ]:
from pathlib import Path

DATASET_PATH = "/content/latency_dataset.csv"
BENCHMARK_PATH = "/content/benchmark_traces.csv"
METADATA_PATH = "/content/collection_metadata.json"

latency_df.to_csv(DATASET_PATH, index=False)
benchmark_df.to_csv(BENCHMARK_PATH, index=False)

metadata = {
    "project": "CAMP - Calibration-Aware ML-Predictive Scheduling",
    "model_id": MODEL_ID,
    "gpu_name": torch.cuda.get_device_name(0),
    "gpu_memory_gib": round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    "vllm_version": __import__("vllm").__version__,
    "seed": SEED,
    "requests_per_run": REQUESTS_PER_RUN,
    "repeats_per_workload": REPEATS_PER_WORKLOAD,
    "concurrency_levels": CONCURRENCY_LEVELS,
    "workloads": WORKLOAD_TYPES,
    "latency_target": "engine_latency_s = completion_time - admission_time",
    "notes": "Pilot dataset. Record actual vLLM version and environment in the final experiment log."
}
Path(METADATA_PATH).write_text(json.dumps(metadata, indent=2))

successful = latency_df[latency_df["status"] == "completed"].copy()
print("Saved:", DATASET_PATH)
print("Saved:", BENCHMARK_PATH)
print("Saved:", METADATA_PATH)
print("\nCollection summary by workload:")
display(successful.groupby("workload_type").agg(
    requests=("request_id", "count"),
    mean_prompt_tokens=("prompt_tokens", "mean"),
    mean_output_tokens=("actual_output_tokens", "mean"),
    mean_engine_latency_s=("engine_latency_s", "mean"),
    p95_engine_latency_s=("engine_latency_s", lambda x: x.quantile(0.95)),
    mean_queue_wait_s=("queue_wait_s", "mean"),
    mean_gpu_utilization_pct=("gpu_utilization_pct", "mean")
).round(3))

if len(successful) < 0.9 * len(latency_df):
    print("WARNING: More than 10% of requests failed. Inspect the error column before using this dataset.")
else:
    print("\nCollection success rate is at least 90%. Inspect errors and telemetry NaNs before training.")

Saved: /content/latency_dataset.csv
Saved: /content/benchmark_traces.csv
Saved: /content/collection_metadata.json

Collection summary by workload:


,requests,mean_prompt_tokens,mean_output_tokens,mean_engine_latency_s,p95_engine_latency_s,mean_queue_wait_s,mean_gpu_utilization_pct
workload_type,,,,,,,
bursty,60,2.0,61.400,1.120,1.747,1.985,77.200
mixed,60,2.0,59.650,1.145,1.824,2.558,96.517
steady,60,2.0,53.333,0.953,1.622,0.935,95.683



Collection success rate is at least 90%. Inspect errors and telemetry NaNs before training.


## Cell 10 — Download the files

Download all three files and keep them for Notebook 02 and Notebook 03. Colab's `/content` storage is temporary, so download them before ending the session.

In [ ]:
from google.colab import files

files.download(DATASET_PATH)
files.download(BENCHMARK_PATH)
files.download(METADATA_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## What to check before moving to Notebook 02

- vLLM server started and the Qwen smoke test generated a response.
- `latency_dataset.csv` contains completed and failed request records.
- `engine_latency_s` and `queue_wait_s` contain sensible non-negative values.
- `gpu_utilization_pct` and `gpu_memory_used_mib` are not entirely missing.
- The dataset has multiple `run_id` values so Notebook 02 can split by run instead of randomly splitting individual rows.
- `benchmark_traces.csv` contains new prompts that were not used to train the predictor.

**Important methodology note:** The measured target is a client/gateway-observed interval from admission to final streamed response. It includes local HTTP overhead and vLLM's internal queueing as well as prefill/decode. It should not be described as pure GPU compute time. The measured predictor is trained on this precisely defined interval.